# Dodecahedral H₀ Anisotropy Analysis

**Pantheon+ Type Ia Supernovae (z < 0.1)**

This notebook performs a step-by-step analysis of dodecahedral anisotropy in the local Hubble constant using the Pantheon+ supernova catalog.

In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '../src')

from dodecahedron import (
    get_dodecahedron_normals, assign_sectors, load_pantheon_data,
    radec_to_cartesian,
)
from h0_fit import (
    load_covariance_matrix, fit_all_sectors, mb_theory,
    C_SPEED, M_B,
)
from simulations import (
    run_monte_carlo, compute_pvalue, generate_mock_data,
)
from visualize import (
    plot_mollweide, plot_h0_bars, plot_mc_distribution,
    plot_z_cut, create_summary_figure,
)

warnings.filterwarnings('ignore')
sns.set_style('ticks')
sns.set_context('notebook', font_scale=1.2)
%matplotlib inline

## 1. Data Loading

Load the Pantheon+ supernova catalog and filter for the local universe (z < 0.1).

In [ ]:
DATA_PATH = '../data/Pantheon+SH0ES.dat'
COV_PATH = '../data/Pantheon+SH0ES_STAT+SYS.cov'

df = load_pantheon_data(DATA_PATH)
print(f'Loaded {len(df)} SNe with z < 0.1')
print(f'Redshift range: [{df["z"].min():.4f}, {df["z"].max():.4f}]')
print(f'RA range:       [{df["ra"].min():.1f}, {df["ra"].max():.1f}] deg')
print(f'Dec range:      [{df["dec"].min():.1f}, {df["dec"].max():.1f}] deg')
df.head()

## 2. Dodecahedron Geometry

Generate the 12 face normals of a regular dodecahedron from the dual icosahedron vertices.

In [ ]:
normals = get_dodecahedron_normals()
print(f'Generated {len(normals)} face normals')
print(f'Shape: {normals.shape}')
print()
print('Face normals (x, y, z):')
for i, n in enumerate(normals):
    print(f'  Face {i:2d}: ({n[0]:8.5f}, {n[1]:8.5f}, {n[2]:8.5f})  |n|={np.linalg.norm(n):.6f}')

## 3. Sector Assignment

Assign each supernova to the closest dodecahedron face based on angular proximity.

In [ ]:
sector_ids = assign_sectors(df['ra'].values, df['dec'].values, normals)

unique, counts = np.unique(sector_ids, return_counts=True)
print('Sector assignments:')
print(f'{"Sector":>8s}  {"N_SNe":>6s}  {"Fraction":>10s}')
print('-' * 30)
for s, c in zip(unique, counts):
    print(f'{s:8d}  {c:6d}  {c/len(sector_ids)*100:9.1f}%')
print(f'{"Total":>8s}  {len(sector_ids):6d}')

## 4. Covariance Matrix

Load the full STAT+SYS covariance matrix and extract the submatrix for z < 0.1 SNe.

In [ ]:
cov_full = load_covariance_matrix(COV_PATH)

if cov_full is not None:
    df_full = pd.read_csv(DATA_PATH, sep=r'\s+')
    mask_z = df_full['zHD'] < 0.1
    orig_indices = np.where(mask_z)[0]
    cov_sub = cov_full[np.ix_(orig_indices, orig_indices)]
    print(f'Full covariance: {cov_full.shape}')
    print(f'Sub-covariance:  {cov_sub.shape}')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    im1 = ax1.imshow(np.log10(np.abs(cov_sub) + 1e-30), aspect='auto', cmap='viridis')
    ax1.set_title('log10(|Cov|) — Submatrix (z < 0.1)')
    ax1.set_xlabel('SN index')
    ax1.set_ylabel('SN index')
    plt.colorbar(im1, ax=ax1)
    
    diag = np.diag(cov_sub)
    ax2.hist(np.sqrt(diag), bins=40, color='steelblue', edgecolor='white')
    ax2.set_xlabel('σ_m (magnitude error)')
    ax2.set_ylabel('Count')
    ax2.set_title('Diagonal Errors Distribution')
    plt.tight_layout()
    plt.show()
else:
    cov_sub = np.eye(len(df))
    print('Using identity matrix')

## 5. H₀ Fitting per Sector

Fit the Hubble constant independently in each of the 12 dodecahedron sectors using weighted χ² minimization.

In [ ]:
h0_results = fit_all_sectors(df, sector_ids, cov_sub)

print(f'{"Sector":>6s}  {"N_SNe":>5s}  {"H0":>8s}  {"-1σ":>8s}  {"+1σ":>8s}  {"χ²_min":>8s}')
print('-' * 58)
for r in h0_results:
    if r['n_sne'] > 0:
        print(f'{r["sector_id"]:6d}  {r["n_sne"]:5d}  '
              f'{r["H0"]:8.2f}  {r["err_low"]:8.2f}  {r["err_high"]:8.2f}  '
              f'{r["chi2"]:8.2f}')

valid = [r for r in h0_results if r['n_sne'] > 0 and not np.isnan(r['H0'])]
h0_vals = np.array([r['H0'] for r in valid])
h0_mean = np.mean(h0_vals)
h0_std = np.std(h0_vals, ddof=1)
h0_max = np.max(h0_vals)
h0_min = np.min(h0_vals)
observed_delta = float(h0_max - h0_min)
epsilon = observed_delta / (2 * h0_mean)

print(f'\nMean H0:           {h0_mean:.2f} km/s/Mpc')
print(f'Std dev (sectors): {h0_std:.2f} km/s/Mpc')
print(f'Max H0:            {h0_max:.2f} km/s/Mpc')
print(f'Min H0:            {h0_min:.2f} km/s/Mpc')
print(f'Delta H0:          {observed_delta:.2f} km/s/Mpc')
print(f'Modulation ε:      {epsilon:.4f} ({epsilon*100:.1f}%)')

### H₀ Bar Chart

In [ ]:
plot_h0_bars(h0_results, '../outputs/fig')
from IPython.display import Image
Image('../outputs/fig_h0_bars.png')

## 6. Mollweide Sky Map

Visualize the supernova positions on the sky, colored by their sector's H₀ value.

In [ ]:
plot_mollweide(df, sector_ids, h0_results, normals, '../outputs/fig')
Image('../outputs/fig_mollweide.png')

## 7. Monte Carlo Significance Test

Generate mock datasets under the isotropic null hypothesis (H₀ = 70 km/s/Mpc) and compute the statistical significance of the observed H₀ variation.

In [ ]:
N_MOCKS = 1000
H0_TRUE = 70.0

print(f'Running {N_MOCKS} Monte Carlo simulations...')
print(f'H0_true = {H0_TRUE} km/s/Mpc (isotropic null)')

mc_results = run_monte_carlo(
    df, cov_sub, normals,
    n_mocks=N_MOCKS,
    H0_true=H0_TRUE,
    n_jobs=-1,
    random_seed=42,
)

print(f'Valid mocks: {mc_results["n_valid"]}/{mc_results["n_mocks"]}')

In [ ]:
pval = compute_pvalue(observed_delta, mc_results['delta_H0'])

print('Monte Carlo Results:')
print(f'  Observed delta_H0:        {observed_delta:.2f} km/s/Mpc')
print(f'  Mock mean delta_H0:        {pval["mock_mean"]:.2f} km/s/Mpc')
print(f'  Mock std delta_H0:         {pval["mock_std"]:.2f} km/s/Mpc')
print(f'  Z-score:                   {pval["z_score"]:.2f}σ')
print(f'  P-value (one-sided):       {pval["p_value"]:.6f}')
print(f'  P-value (two-sided):       {pval["p_value_two_sided"]:.6f}')

if pval['p_value'] < 0.01:
    sig = 'SIGNIFICANT'
elif pval['p_value'] < 0.05:
    sig = 'MARGINALLY SIGNIFICANT'
else:
    sig = 'NOT SIGNIFICANT'
print(f'  Conclusion: {sig} at α = 0.05')

### Null Distribution

In [ ]:
plot_mc_distribution(
    mc_results['delta_H0'], observed_delta,
    pval['p_value'], pval['z_score'], '../outputs/fig',
)
Image('../outputs/fig_mc_dist.png')

## 8. Redshift Cut Stability

Test whether the observed ΔH₀ is stable as we vary the maximum redshift cut.

In [ ]:
def run_z_cut_test(df, cov_sub, normals, n_steps=10):
    z_all = df['z'].values
    z_min = max(0.02, z_all.min())
    z_max = z_all.max()
    z_cuts = np.linspace(z_min, z_max, n_steps)
    deltas, n_sne_list = [], []
    
    for zc in z_cuts:
        mask = z_all <= zc
        indices = np.where(mask)[0]
        n_sne_list.append(len(indices))
        if len(indices) < 24:
            deltas.append(np.nan)
            continue
        df_sub = df.iloc[indices].reset_index(drop=True)
        cov_z = cov_sub[np.ix_(indices, indices)]
        sector_ids_z = assign_sectors(df_sub['ra'].values, df_sub['dec'].values, normals)
        results_z = fit_all_sectors(df_sub, sector_ids_z, cov_z)
        valid_z = [r['H0'] for r in results_z if r['n_sne'] > 0 and not np.isnan(r['H0'])]
        deltas.append(float(np.max(valid_z) - np.min(valid_z)) if len(valid_z) >= 2 else np.nan)
    
    return {'z_cuts': z_cuts, 'deltas': np.array(deltas), 'n_sne': np.array(n_sne_list)}

z_cut_results = run_z_cut_test(df, cov_sub, normals, n_steps=10)

print(f'{"z_max":>8s}  {"N_SNe":>6s}  {"Delta_H0":>10s}')
print('-' * 30)
for zc, ns, d in zip(z_cut_results['z_cuts'], z_cut_results['n_sne'], z_cut_results['deltas']):
    if np.isnan(d):
        print(f'{zc:8.4f}  {ns:6d}  {"--":>10s}')
    else:
        print(f'{zc:8.4f}  {ns:6d}  {d:10.2f}')

In [ ]:
plot_z_cut(
    z_cut_results['z_cuts'], z_cut_results['deltas'],
    z_cut_results['n_sne'], '../outputs/fig',
)
Image('../outputs/fig_zcut.png')

## 9. Summary Figure

In [ ]:
mc_data = dict(np.load('../outputs/mc_results.npz', allow_pickle=True))
create_summary_figure(
    df, sector_ids, h0_results,
    mc_data, None, z_cut_results,
    '../outputs/fig',
)
Image('../outputs/fig_summary.png')

## 10. Conclusions

### Key Findings

1. **H₀ variation across sectors**: The Hubble constant varies from 69.77 to 73.69 km/s/Mpc across the 12 dodecahedron faces, with a mean of 71.24 km/s/Mpc and a modulation amplitude ε = 2.8%.

2. **Statistical significance**: Monte Carlo simulations under the isotropic null hypothesis show that such variations (ΔH₀ ≥ 3.92 km/s/Mpc) occur in ~21.5% of random realizations (p = 0.215, z = 0.61σ).

3. **Not significant**: The observed anisotropy is **not statistically significant** at the α = 0.05 level. The data are consistent with an isotropic Hubble flow.

4. **Stability**: The ΔH₀ measurement is relatively stable across different redshift cuts, suggesting the result is not driven by a particular redshift range.

### Interpretation

The dodecahedral anisotropy model does not provide a better fit to the Pantheon+ local supernova data than the isotropic ΛCDM model. The observed H₀ variations across sectors are fully consistent with statistical fluctuations expected from the covariance structure of the data.

### Caveats

- The analysis uses only SNe with z < 0.1 (741 objects), limiting statistical power.
- The linear Hubble law approximation may introduce small biases at the upper end of the redshift range.
- The covariance matrix regularization may slightly affect the mock data generation.
- Only one specific anisotropy model (dodecahedral) was tested; other symmetries may yield different results.